# ЛР 03.1 — Train/Validation и переобучение (solution)

## Цель
- взять лучший неполный feature set из ЛР 01 как стартовую точку;
- сравнить `full` и неполный feature set на `train` и `validation`;
- увидеть `generalization gap` на понятных цифрах;
- посмотреть, как один гиперпараметр меняет поведение модели.

## Что важно
- высокая метрика на `train` еще не означает хорошую работу на новых данных;
- `test` пока не используется для выбора модели;
- здесь мы ищем честные сигналы переобучения, а не самую красивую train-цифру.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from IPython.display import display
from sklearn.base import clone

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '03-overfitting-validation-and-hyperparameter-tuning',
    cwd.parent / '03-overfitting-validation-and-hyperparameter-tuning',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        'Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 03 или из корня репозитория.'
    )

spec = importlib.util.spec_from_file_location('lab03_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


## Шаг 1. Загрузка данных и стартового feature set из ЛР 01

На этом шаге:
- загружаем оба датасета курса;
- берем лучший неполный feature set по артефактам ЛР 01;
- один раз делим данные на `train`, `validation`, `test` в схеме `60/20/20`.

`test` пока откладываем в сторону: он понадобится только в конце второго ноутбука.


In [ ]:
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

prepared = {}
selection_rows = []

for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_valid, x_test, y_train, y_valid, y_test = lab.train_valid_test_split_stratified(x, y)

    starter_feature_set = lab.choose_best_nonfull_feature_set(
        model_results=model_results,
        feature_sets=feature_sets,
        dataset_name=dataset_name,
    )
    selected_features = feature_sets[dataset_name][starter_feature_set]
    category_levels = lab.infer_category_levels(x_train)

    full_selector = lab.PreprocessedFeatureSelector(
        selected_features=None,
        category_levels=category_levels,
    ).fit(x_train, y_train)
    selected_selector = lab.PreprocessedFeatureSelector(
        selected_features=selected_features,
        category_levels=category_levels,
    ).fit(x_train, y_train)

    full_feature_names = full_selector.get_feature_names_out().tolist()
    selected_feature_names = selected_selector.get_feature_names_out().tolist()

    prepared[dataset_name] = {
        'x_train': x_train,
        'x_valid': x_valid,
        'x_test': x_test,
        'y_train': y_train,
        'y_valid': y_valid,
        'y_test': y_test,
        'starter_feature_set': starter_feature_set,
        'selected_features': selected_feature_names,
        'category_levels': category_levels,
        'feature_sets': {
            'full': full_feature_names,
            starter_feature_set: selected_feature_names,
        },
        'matrix_cache': {
            'full': {
                'x_train': full_selector.transform(x_train),
                'x_valid': full_selector.transform(x_valid),
            },
            starter_feature_set: {
                'x_train': selected_selector.transform(x_train),
                'x_valid': selected_selector.transform(x_valid),
            },
        },
    }

    selection_rows.append(
        {
            'dataset': dataset_name,
            'starter_feature_set': starter_feature_set,
            'n_train': len(x_train),
            'n_validation': len(x_valid),
            'n_test': len(x_test),
            'n_full_features': len(full_feature_names),
            'n_selected_features': len(selected_feature_names),
            'selected_preview': ', '.join(selected_feature_names[:5]),
        }
    )

selection_summary = pd.DataFrame(selection_rows)
selection_summary


## Шаг 2. Сравнение базовых моделей на `train` и `validation`

Сравниваем только две модели:
- `LogisticRegression`
- `RandomForestClassifier`

Для каждой модели считаем:
- `accuracy`
- `f1`
- `roc_auc`
- `fit_time_sec`


In [ ]:
audit_rows = []

for dataset_name, ctx in prepared.items():
    for feature_set_name, matrices in ctx['matrix_cache'].items():
        for model_name, model in lab.make_default_models().items():
            _, fit_time_sec, train_metrics, valid_metrics = lab.measure_fit_and_split_metrics(
                clone(model),
                matrices['x_train'],
                ctx['y_train'],
                matrices['x_valid'],
                ctx['y_valid'],
            )

            for split_name, metrics in [
                ('train', train_metrics),
                ('validation', valid_metrics),
            ]:
                audit_rows.append(
                    {
                        'dataset': dataset_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'split': split_name,
                        'accuracy': metrics['accuracy'],
                        'f1': metrics['f1'],
                        'roc_auc': metrics['roc_auc'],
                        'fit_time_sec': fit_time_sec,
                    }
                )

generalization_audit = (
    pd.DataFrame(audit_rows)
    .sort_values(['dataset', 'feature_set', 'model', 'split'])
    .reset_index(drop=True)
)
generalization_audit


## Шаг 3. Где виден `generalization gap`

Здесь важно не просто увидеть абсолютную метрику, а сопоставить:
- насколько хорошо модель выглядит на `train`;
- насколько это качество переносится на `validation`.

Если разрыв большой, это уже сигнал переобучения.


In [ ]:
gap_summary = lab.build_generalization_selection_summary(generalization_audit).reset_index(drop=True)

feature_set_decision_rows = []
for dataset_name, ctx in prepared.items():
    feature_set_for_notebook_02 = lab.choose_lab03_feature_set(generalization_audit, dataset_name)
    feature_set_decision_rows.append(
        {
            'dataset': dataset_name,
            'starter_feature_set_from_lab01': ctx['starter_feature_set'],
            'feature_set_for_notebook_02': feature_set_for_notebook_02,
            'primary_rule': 'max validation f1',
            'secondary_rule': 'min f1 gap',
        }
    )

feature_set_decisions = pd.DataFrame(feature_set_decision_rows)
display(gap_summary)
feature_set_decisions


## Шаг 4. Простые validation curves

Мы не строим полный поиск параметров в этом ноутбуке.
Вместо этого смотрим на один параметр за раз:
- для `LogisticRegression` меняем `C`;
- для `RandomForest` меняем `max_depth`.

Это помогает увидеть, где модель становится слишком простой, а где уже начинает переобучаться.


In [ ]:
curve_rows = []

for dataset_name, ctx in prepared.items():
    feature_set_name = lab.choose_lab03_feature_set(generalization_audit, dataset_name)
    matrices = ctx['matrix_cache'][feature_set_name]

    for model_name, base_model in lab.make_default_models().items():
        hyperparameter, param_grid = lab.VALIDATION_CURVE_GRIDS[model_name]
        for param_value in param_grid:
            model = clone(base_model)
            model.set_params(**{hyperparameter: param_value})
            _, _, train_metrics, valid_metrics = lab.measure_fit_and_split_metrics(
                model,
                matrices['x_train'],
                ctx['y_train'],
                matrices['x_valid'],
                ctx['y_valid'],
            )

            for split_name, metrics in [
                ('train', train_metrics),
                ('validation', valid_metrics),
            ]:
                curve_rows.append(
                    {
                        'dataset': dataset_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'hyperparameter': hyperparameter,
                        'param_value': lab.format_param_value(param_value),
                        'split': split_name,
                        'accuracy': metrics['accuracy'],
                        'f1': metrics['f1'],
                        'roc_auc': metrics['roc_auc'],
                    }
                )

validation_curve_results = (
    pd.DataFrame(curve_rows)
    .sort_values(['dataset', 'model', 'split', 'param_value'])
    .reset_index(drop=True)
)
validation_curve_results.head(20)


## Шаг 5. Быстрый визуальный обзор

Ниже не нужно искать "идеальную линию".
Смысл графиков в том, чтобы глазами увидеть:
- где train и validation расходятся;
- где усложнение модели уже не помогает validation-качеству.


In [ ]:
plot_df = validation_curve_results.copy()

grid = sns.relplot(
    data=plot_df,
    x='param_value',
    y='f1',
    hue='split',
    col='model',
    row='dataset',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': False},
)
grid.set_axis_labels('Значение гиперпараметра', 'F1')
grid.set_titles(row_template='{row_name}', col_template='{col_name}')
plt.show()


## Самостоятельное изучение по ходу работы

Заполните своими словами:
- где вы увидели самый явный пример переобучения;
- в каком случае train-метрика выглядела слишком оптимистично;
- как вы объясните новичку разницу между `train`, `validation` и `test`.

При необходимости вынесите разбор в:
- `study-notes/overfitting-vs-underfitting.md`
- `study-notes/train-validation-test-split.md`


## Контрольные точки

Перед переходом ко второму ноутбуку проверьте:
1. Есть таблица `generalization_audit` с раздельными метриками для `train` и `validation`.
2. Есть таблица `validation_curve_results`.
3. Для каждого датасета вы выбрали feature set для следующего шага по правилу:
   `max validation f1 -> min f1 gap -> prefer non-full on tie`.
4. `test` еще не использовался для выбора модели или параметров.


In [ ]:
required_generalization_columns = {
    'dataset',
    'feature_set',
    'model',
    'split',
    'accuracy',
    'f1',
    'roc_auc',
    'fit_time_sec',
}
required_curve_columns = {
    'dataset',
    'feature_set',
    'model',
    'hyperparameter',
    'param_value',
    'split',
    'accuracy',
    'f1',
    'roc_auc',
}

assert set(required_generalization_columns).issubset(generalization_audit.columns)
assert set(required_curve_columns).issubset(validation_curve_results.columns)

generalization_audit_path = OUTPUT_DIR / 'generalization_audit.csv'
validation_curve_results_path = OUTPUT_DIR / 'validation_curve_results.csv'

generalization_audit.to_csv(generalization_audit_path, index=False)
validation_curve_results.to_csv(validation_curve_results_path, index=False)

print('Saved:', generalization_audit_path)
print('Saved:', validation_curve_results_path)
